In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

import s3_utils

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

In [ ]:
print('Loading Winogender DLA results and metadata...')
prob_df = s3_utils.read_csv('outputs/gpt2-xl/winogender/out_DLA_winogender.csv')
impact_df = s3_utils.read_csv('outputs/gpt2-xl/winogender/accumulated_impact_winogender.csv')
metadata = s3_utils.read_json('data/winogender/winogender_metadata.json')

meta_df = pd.DataFrame(metadata)

prob_df = prob_df.merge(meta_df, left_on='ID', right_on='id', how='left')
impact_df = impact_df.merge(meta_df, left_on='ID', right_on='id', how='left')

print(f'DLA rows: {len(prob_df)}, Impact rows: {len(impact_df)}')
print(f'Unique IDs: {prob_df["ID"].nunique()}')
print(f'  answer=0 (pronoun=occupation): {meta_df[meta_df["answer"]==0].shape[0]}')
print(f'  answer=1 (pronoun=participant): {meta_df[meta_df["answer"]==1].shape[0]}')

In [ ]:
def compute_metrics(prob_data):
    """Compute SS, LMS, ICAT from a probability DataFrame (same logic as StereoSet)."""
    max_layer = prob_data['Layer'].max()
    last_tok_idx = prob_data.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
    final = prob_data.loc[last_tok_idx]
    final = final[final['Layer'] == max_layer]
    pivot = final.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

    n_total = len(pivot)
    related = 0
    n_stereo = 0
    n_anti = 0

    has_all_three = {'stereotype', 'anti-stereotype', 'unrelated'} <= set(pivot.columns)
    has_stereo_anti = {'stereotype', 'anti-stereotype'} <= set(pivot.columns)

    if has_all_three:
        related += int((pivot['stereotype'] > pivot['unrelated']).sum())
        related += int((pivot['anti-stereotype'] > pivot['unrelated']).sum())
    if has_stereo_anti:
        n_stereo = int((pivot['stereotype'] > pivot['anti-stereotype']).sum())
        n_anti = int((pivot['anti-stereotype'] > pivot['stereotype']).sum())

    lms = (related / (2 * n_total) * 100) if (n_total > 0 and has_all_three) else 0.0
    denom = n_stereo + n_anti
    ss = (n_stereo / denom * 100) if denom > 0 else 50.0
    icat = lms * (min(ss, 100.0 - ss) / 50.0)

    return ss, lms, icat

In [ ]:
prob_occ = prob_df[prob_df['answer'] == 0].copy()
prob_part = prob_df[prob_df['answer'] == 1].copy()

ss_all, lms_all, icat_all = compute_metrics(prob_df)
ss_occ, lms_occ, icat_occ = compute_metrics(prob_occ)
ss_part, lms_part, icat_part = compute_metrics(prob_part)

summary = pd.DataFrame([
    {'Subset': 'All (120 templates)', 'SS': ss_all, 'LMS': lms_all, 'ICAT': icat_all},
    {'Subset': 'answer=0 (pronoun=occupation)', 'SS': ss_occ, 'LMS': lms_occ, 'ICAT': icat_occ},
    {'Subset': 'answer=1 (pronoun=participant)', 'SS': ss_part, 'LMS': lms_part, 'ICAT': icat_part},
])

print('Winogender Baseline Metrics (GPT-2 XL):')
print('  answer=0 = primary bias measure (pronoun refers to occupation)')
  
print('  answer=1 = control group (pronoun refers to participant)\n')
display(summary.style.format({'SS': '{:.2f}', 'LMS': '{:.2f}', 'ICAT': '{:.2f}'}).hide(axis='index'))

In [ ]:
max_layer = prob_occ['Layer'].max()
last_tok = prob_occ.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
final_occ = prob_occ.loc[last_tok]
final_occ = final_occ[final_occ['Layer'] == max_layer]

pivot_occ = final_occ.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)
pivot_occ = pivot_occ.merge(
    meta_df[meta_df['answer'] == 0][['id', 'occupation', 'bls_pct_female']],
    left_index=True, right_on='id'
).set_index('id')

pivot_occ['bias_score'] = pivot_occ['stereotype'] - pivot_occ.get('anti-stereotype', 0)
pivot_occ['gender_majority'] = pivot_occ['bls_pct_female'].apply(
    lambda x: 'Female-dominated (>50%)' if x > 50 else 'Male-dominated (<=50%)'
)

occ_bias = pivot_occ.sort_values('bias_score')

fig, ax = plt.subplots(figsize=(10, 16))
colors = occ_bias['gender_majority'].map({
    'Female-dominated (>50%)': '#e74c3c',
    'Male-dominated (<=50%)': '#3498db'
})
ax.barh(occ_bias['occupation'], occ_bias['bias_score'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Bias Score: P(stereotype) - P(anti-stereotype)', fontsize=12)
ax.set_title('Per-Occupation Gender Bias Score (answer=0, GPT-2 XL Baseline)', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Female-dominated (>50%)'),
    Patch(facecolor='#3498db', label='Male-dominated (<=50%)')
]
ax.legend(handles=legend_elements, loc='lower right')
sns.despine(ax=ax)
plt.tight_layout()
s3_utils.save_plot(fig, 'outputs/gpt2-xl/winogender/plots/per_occupation_bias_score.pdf')
plt.show()

In [ ]:
# For BLS correlation, compute raw P(she) - P(he) regardless of stereotype labeling
# We need to go back to the raw candidate probabilities
final_occ_raw = prob_occ.loc[
    prob_occ.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
]
final_occ_raw = final_occ_raw[final_occ_raw['Layer'] == max_layer]

# Get per-ID candidate probabilities by looking at the actual Candidate text
# (the Type column says stereotype/anti-stereotype, but Candidate has the actual pronoun)
cand_pivot = final_occ_raw.pivot(
    index='ID', columns='Candidate', values='Layer_Accumulated_Prob'
).fillna(0)

cand_pivot = cand_pivot.merge(
    meta_df[meta_df['answer'] == 0][['id', 'occupation', 'bls_pct_female']],
    left_index=True, right_on='id'
).set_index('id')

# Determine which pronoun columns are present (depends on pronoun_type: nom/poss/acc)
# Group by occupation to average over the two answer types per occupation
# For nominative: he/she, possessive: his/her, accusative: him/her
female_pronouns = {'she', 'her', 'her'}  # 'her' covers both poss and acc
male_pronouns = {'he', 'his', 'him'}

female_cols = [c for c in cand_pivot.columns if c in female_pronouns]
male_cols = [c for c in cand_pivot.columns if c in male_pronouns]

cand_pivot['p_female'] = cand_pivot[female_cols].sum(axis=1) if female_cols else 0
cand_pivot['p_male'] = cand_pivot[male_cols].sum(axis=1) if male_cols else 0
cand_pivot['female_minus_male'] = cand_pivot['p_female'] - cand_pivot['p_male']

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(cand_pivot['bls_pct_female'], cand_pivot['female_minus_male'],
           alpha=0.7, s=60, edgecolor='black', linewidth=0.5)

for _, row in cand_pivot.iterrows():
    ax.annotate(row['occupation'], (row['bls_pct_female'], row['female_minus_male']),
                fontsize=7, alpha=0.7, ha='center', va='bottom')

x = cand_pivot['bls_pct_female'].values
y = cand_pivot['female_minus_male'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r--', linewidth=2,
        label=f'OLS: r={r_value:.3f}, p={p_value:.2e}')

ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('BLS % Female in Occupation', fontsize=12)
ax.set_ylabel('P(female pronoun) - P(male pronoun)', fontsize=12)
ax.set_title('Model Pronoun Preference vs Real-World Gender Statistics\n(answer=0 templates, GPT-2 XL)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
sns.despine(ax=ax)
plt.tight_layout()
s3_utils.save_plot(fig, 'outputs/gpt2-xl/winogender/plots/bls_correlation.pdf')
plt.show()

print(f'Pearson r = {r_value:.4f}, p-value = {p_value:.4e}')
print(f'Slope = {slope:.6f} (per 1% increase in BLS female share)')

In [ ]:
def generate_grouped_analysis(sub_df, scenario_title, all_probs_df, winner_type):
    """3-panel impact plot: MLP impact, Head sum, Head heatmap."""
    if len(sub_df) == 0:
        return

    relevant_ids = sub_df['ID'].unique()
    current_probs = all_probs_df[all_probs_df['ID'].isin(relevant_ids)].copy()

    type_map = {'stereotype': 'Stereotype', 'anti-stereotype': 'Anti-Stereotype', 'unrelated': 'Unrelated'}
    current_probs['Type'] = current_probs['Type'].map(type_map).fillna(current_probs['Type'])
    plot_df = sub_df.copy()
    plot_df['Type'] = plot_df['Type'].map(type_map).fillna(plot_df['Type'])
    winner_label = type_map.get(winner_type, winner_type)

    mlp_df = plot_df[plot_df['Component'] == 'MLP']
    mlp_summary = mlp_df.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    prob_summary = current_probs.groupby(['Layer', 'Type'])['Layer_Accumulated_Prob'].mean().reset_index()
    prob_pivot = prob_summary.pivot(index='Layer', columns='Type', values='Layer_Accumulated_Prob').fillna(0)
    prob_summary_winner = pd.DataFrame({'Layer': prob_pivot.index})

    if winner_label == 'Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Stereotype', 0) - prob_pivot.get('Anti-Stereotype', 0)
        y_label = 'Margin: P(Stereo) - P(Anti)'
    elif winner_label == 'Anti-Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Anti-Stereotype', 0) - prob_pivot.get('Stereotype', 0)
        y_label = 'Margin: P(Anti) - P(Stereo)'
    else:
        prob_summary_winner['Margin'] = prob_pivot.get(winner_label, 0)
        y_label = f'Prob ({winner_label})'

    head_df = plot_df[plot_df['Component'].str.startswith('Head')].copy()
    head_df['Head_ID'] = head_df['Component'].str.replace('Head_', '').astype(int)
    heatmap_data = head_df[head_df['Type'] == winner_label]
    head_matrix = heatmap_data.groupby(['Head_ID', 'Layer'])['Accumulated_Impact'].mean().unstack()

    head_layer_sum = head_df.groupby(['ID', 'Layer', 'Type'])['Accumulated_Impact'].sum().reset_index()
    head_sum_summary = head_layer_sum.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.9], hspace=0.3, wspace=0.25)
    ax_mlp = fig.add_subplot(gs[0, 0])
    ax_sum = fig.add_subplot(gs[0, 1])
    ax_heat = fig.add_subplot(gs[1, :])

    ax_mlp_twin = ax_mlp.twinx()
    sns.barplot(data=prob_summary_winner, x='Layer', y='Margin',
                color='lightgray', alpha=0.5, ax=ax_mlp_twin, errorbar=None)
    ax_mlp_twin.set_ylabel(y_label, color='gray', fontweight='bold')
    ax_mlp_twin.grid(False)
    ax_mlp_twin.set_xticks([])

    sns.lineplot(data=mlp_summary, x='Layer', y='Accumulated_Impact', hue='Type',
                 marker='o', linewidth=2.5, ax=ax_mlp, palette='colorblind')
    ax_mlp.set_title('MLP Impact & Probability Margin', fontweight='bold')
    ax_mlp.set_ylabel('Cumulative Logit Contribution')
    ax_mlp.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_mlp.legend(title='', loc='upper left', frameon=False)
    sns.despine(ax=ax_mlp, right=False)

    sns.lineplot(data=head_sum_summary, x='Layer', y='Accumulated_Impact', hue='Type',
                 marker='s', linewidth=2.5, ax=ax_sum, palette='colorblind')
    ax_sum.set_title('Total Attention Block Impact (Sum of Heads)', fontweight='bold')
    ax_sum.set_ylabel('Cumulative Logit Contribution')
    ax_sum.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_sum.legend(title='', loc='upper left', frameon=False)
    sns.despine(ax=ax_sum)

    if head_matrix is not None and not head_matrix.empty:
        sns.heatmap(head_matrix, cmap='RdBu_r', center=0, ax=ax_heat,
                    xticklabels=5, yticklabels=2, cbar_kws={'label': 'Accumulated Impact'})
        ax_heat.set_title(f'Per-Head Impact Heatmap ({winner_label})', fontweight='bold')
        ax_heat.set_xlabel('Layer')
        ax_heat.set_ylabel('Head')

    plt.suptitle(scenario_title, fontsize=18, fontweight='bold', y=0.96)

    safe_title = scenario_title.replace(' ', '_').replace('(', '').replace(')', '').replace(':', '').replace('%', '').replace('\n', '_')
    s3_utils.save_plot(fig, f'outputs/gpt2-xl/winogender/plots/{safe_title}_impact_analysis.pdf')
    plt.show()
    plt.close(fig)


# Top-5 most biased occupations by |bias_score|
top5 = occ_bias.reindex(occ_bias['bias_score'].abs().sort_values(ascending=False).index).head(5)

for _, row in top5.iterrows():
    occ = row['occupation']
    occ_ids = meta_df[(meta_df['answer'] == 0) & (meta_df['occupation'] == occ)]['id'].values
    sub_impact = impact_df[impact_df['ID'].isin(occ_ids)]
    sub_probs = prob_df[prob_df['ID'].isin(occ_ids)]

    direction = 'stereotype' if row['bias_score'] > 0 else 'anti-stereotype'
    title = f'{occ.title()} (bias={row["bias_score"]:.4f}, BLS {row["bls_pct_female"]:.0f}% female)'
    generate_grouped_analysis(sub_impact, title, sub_probs, direction)

In [ ]:
# Optional: compare baseline vs fine-tuned models on Winogender
ft_keys = s3_utils.list_keys('outputs/gpt2-xl/winogender/finetuned/')
prefix = s3_utils.s3_key('outputs/gpt2-xl/winogender/finetuned/')

ft_run_ids = set()
for k in ft_keys:
    parts = k[len(prefix):].split('/')
    if len(parts) >= 1 and parts[0]:
        ft_run_ids.add(parts[0])

ft_run_ids = sorted(ft_run_ids)
print(f'Found {len(ft_run_ids)} fine-tuned Winogender evaluations.')

if ft_run_ids:
    comparison_rows = [{
        'run_id': 'baseline',
        'SS_all': ss_all, 'LMS_all': lms_all, 'ICAT_all': icat_all,
        'SS_occ': ss_occ, 'LMS_occ': lms_occ, 'ICAT_occ': icat_occ,
        'SS_part': ss_part, 'LMS_part': lms_part, 'ICAT_part': icat_part,
    }]

    for rid in ft_run_ids:
        try:
            ft_prob = s3_utils.read_csv(f'outputs/gpt2-xl/winogender/finetuned/{rid}/out_DLA_winogender.csv')
            ft_prob = ft_prob.merge(meta_df, left_on='ID', right_on='id', how='left')

            s_all, l_all, i_all = compute_metrics(ft_prob)
            s_occ, l_occ, i_occ = compute_metrics(ft_prob[ft_prob['answer'] == 0])
            s_part, l_part, i_part = compute_metrics(ft_prob[ft_prob['answer'] == 1])

            comparison_rows.append({
                'run_id': rid,
                'SS_all': s_all, 'LMS_all': l_all, 'ICAT_all': i_all,
                'SS_occ': s_occ, 'LMS_occ': l_occ, 'ICAT_occ': i_occ,
                'SS_part': s_part, 'LMS_part': l_part, 'ICAT_part': i_part,
            })
        except Exception as e:
            print(f'  Skipping {rid}: {e}')

    comp_df = pd.DataFrame(comparison_rows)
    print('\nBaseline vs Fine-tuned (Winogender):')
    display(comp_df.style.format({
        c: '{:.2f}' for c in comp_df.columns if c != 'run_id'
    }).hide(axis='index'))
else:
    print('No fine-tuned Winogender evaluations found. Run winogender_bias_search.py --run_id <id> first.')